---
format:
  html:
    code-fold: true
    code-summary: "Mostrar código"
---

# Introducción a los rendimientos queseros

La elaboración de queso tiene como objetivo principal **preservar y concentrar los componentes nutritivos de la leche**, transformándolos en un producto alimentario con **elevado valor sensorial, nutricional y comercial**. Este proceso da lugar a una amplia variedad de quesos, cada uno con características únicas que responden a factores como el tipo de leche, la tecnología aplicada y las condiciones de maduración.

El propósito de la tecnología quesera es **optimizar esta transformación**, guiando el proceso de forma que se obtenga el mejor producto posible, **garantizando la calidad sanitaria, tecnológica y organoléptica**, y todo ello de manera **eficiente y rentable**. Para lograrlo, los objetivos técnicos se centran en **minimizar las pérdidas de materia**, especialmente de **grasa y proteína**, que son los componentes clave en la formación del queso. Al mismo tiempo, se busca **preservar los atributos sensoriales** —como sabor, textura y aroma— y **cumplir con los requisitos higiénico-sanitarios** establecidos por la normativa vigente.

El análisis del proceso quesero se estructura en dos partes diferenciadas:

-   **Balance de materia global:** comprende desde la recepción de la leche cruda hasta el final del proceso de elaboración, momento en que el queso está listo para entrar en la fase de maduración. Este balance permite evaluar el rendimiento general y detectar posibles pérdidas o desviaciones.
-   **Transformación quesera propiamente dicha:** incluído en el balance de materia global, abarca el tramo desde la leche destinada a fabricación una vez en la cuba, hasta la obtención del queso fresco (antes o después del salado). Aquí se analizan los rendimientos específicos, la eficiencia de coagulación, el corte, el desuerado, el moldeado y otros parámetros tecnológicos que inciden directamente en la calidad y cantidad del producto final.

En este capítulo abordaremos la segunda parte, la transformación quesera, analizando los valores analíticos de la leche de fabricación y del queso recién elaborado, y extraeremos diversas consecuencias:

-   Estableceremos los parámetros de referencia de nuestro propio proceso, que nos servirán de punto de comparación para las fabricaciones que vayamos realizando.
-   Evaluaremos las causas posibles de las desviaciones, y veremos cómo proponer planes de acción para la reducción o eliminación de estas desviaciones.


## Un ejemplo de cálculo de rendimientos

En este ejemplo, vamos a suponer que elaboramos un queso de vaca, del que tenemos valores analíticos de producto terminado y de la leche utilizada en su elaboración. El proceso siempre tiene los mismos pasos tecnológicos, con la variabilidad que suponemos que es la habitual. 

Podemos imaginar dos situaciones en las que nos encontremos con la necesidad de analizar unos datos acumulados:

- Llegamos a una empresa en la que han estado acumulando datos de producción, pero no han sabido cómo formalizar y estructurar los datos adecuadamente
- Estamos trabajando en una empresa en la que se analizan los datos, y se plantean establecer un presupuesto más formal para el siguiente año, lo que exige el análisis de los datos del año en curso y el establecimiento de las hipótesis de presupuesto, entre ellas, el consumo de materia, que determinará las necesidades de aprovisionamiento y el precio de venta del queso.

En ambos casos, necesitaremos analizar un conjunto de datos de fabricación; siguiendo nuestra línea de trabajo, lo haremos con `python`y Excel.

Los datos técnicos de ejemplo que vamos a utilizar están registrados en un fichero de datos en formato `csv`, que incluye:

1.  las analíticas del queso, y
2.  las analíticas de la leche con la que lo hacemos.

Caracterizaremos nuestro queso mediante un análisis de extracto seco total (EST) y una grasa butirométrica (MG), en un punto determinado de nuestro proceso (por ejemplo, a la salida de la salmuera, después de escurrido) y mediante un muestreo repetible, de forma que nuestro análisis sea representativo. A partir de estos dos análisis, calculamos otros parámetros técnicos necesarios, tales como la materia grasa sobre el extracto seco total (G/ES), el extracto seco magro o desnatado (ESM) y la humedad del queso desnatado (HQD), valores que se calculan a partir de los primeros.

En el recorrido de cálculo utilizaremos `Python`, al final se proporciona una hoja de cálculo con la información necesaria.

Como siempre, es opcional descargar los datos de ejemplo y abrir el cuaderno en Colab para ejecutar el código a medida que se avanza en el estudio.

[Abrir este cuaderno en Google Colab](https://colab.research.google.com/github/juanriera/master-queseria/blob/master/085-intro-rendim.ipynb){target="_blank" rel="noopener noreferrer"}

[Descargar los datos de ejemplo utilizados en este cuaderno (archivo `fab_queso_bm.csv`)](https://raw.githubusercontent.com/juanriera/master-queseria/master/datos/fab_queso_bm.csv){target="_blank" rel="noopener noreferrer"}

## Lectura y exploración de los datos

Antes de lanzarnos al análisis cuantitativo, siempre es conveniente una primera visualización de los datos, por si hubiese algún valor que nos pueda resultar cuestionable. 

Es una mala práctica, aunque muy extendida, usar la hoja de cálculo para hacer la media aritmética de cada parámetro que vayamos a utilizar y trabajar con estos valores medios. En un conjunto de datos que recoja muchas fabricaciones, nunca podemos estar seguros de que el 100% de los valores sean correctos, que no haya habido errores de muestreo, analíticos o problemas de fabricación que hayan producido valores anormales. Por eso vamos a utilizar siempre los valores *medianos* y no los valores medios; como sabemos, la **mediana** es resistente a los valores anormales y extremos, mientras que la media no lo es.

Procedemos a leer los datos, y visualizamos las primeras lineas. Se incluye el código `python`que permite visualizar las tablas, en algunos casos se han formateado mediante `HTML`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

url_datos = 'https://raw.githubusercontent.com/juanriera/master-queseria/master/datos/fab_queso_bm.csv'

try:
    df = pd.read_csv(url_datos, decimal = ",", sep=';', encoding='ISO-8859-1')
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

# crear index con fecha
df['fecha_index'] = pd.to_datetime(
    df['fecha'], 
    format='%d/%m/%Y',  # Formato Día/Mes/Año confirmado
    errors='coerce'     # Clave: convierte los valores que no puede leer a NaT (Not a Time)
)
# 2. Limpiar NaT y establecer el índice
df.dropna(subset=['fecha_index'], inplace=True)
df.set_index('fecha_index', inplace=True)
df.sort_index(inplace=True)

# Filtramos sólo queso de vaca
df = df[df['receta']=='vaca']

df.head()

,fecha,receta,litros_past,est_past,mg_past,mp_past,kg_cuba,est_cuba,mg_cuba,mp_cuba,formato,est_salida_salmuera,mg_salida_salmuera,ph_salida_salmuera,peso_queso_total,sal_queso
fecha_index,,,,,,,,,,,,,,,,
2022-03-01,01/03/2022,vaca,14910,12.542,3.994,3.076,15810.99,11.37,4.02,2.78,barra,55.94,30.50,5.41,1959.76,1.55
2022-03-04,04/03/2022,vaca,14100,13.312,4.002,3.673,14989.74,13.25,4.02,3.78,barra,54.58,29.80,5.88,1888.48,1.77
2022-03-25,25/03/2022,vaca,13310,12.856,3.889,3.372,13011.79,12.89,4.02,3.35,barra,54.10,30.63,5.48,1555.84,1.32
2022-04-01,01/04/2022,vaca,15430,13.151,3.886,3.218,15946.80,11.88,3.76,3.42,barra,56.76,30.51,5.36,1884.96,1.36
2022-04-08,08/04/2022,vaca,15930,13.103,4.022,3.396,16452.76,11.93,3.99,3.27,barra,55.63,30.01,5.31,1955.36,1.90


Los datos se corresponden con un conjunto de fabricaciones de queso de vaca, como hemos dicho. LAs principales variables recogidas son las siguientes:

In [2]:
# Diccionario de variables con espacio para completar descripciones
descripcion_variables = {
    'fecha': 'Fecha de producción',
    'receta': 'Tipo de receta utilizada',
    'litros_past': 'Litros de leche pasteurizada',
    'est_past': 'Extracto seco total de la leche pasteurizada (g/100 ml leche)',
    'mg_past': 'Materia grasa de la leche pasteurizada (g/100 ml leche)',
    'mp_past': 'Materia proteica de la leche pasteurizada (g/100 ml leche)',
    'kg_cuba': 'Kilos totales de leche en la cuba',
    'est_cuba': 'Extracto seco total en cuba (g/100 g leche)',
    'mg_cuba': 'Materia grasa en cuba (g/100 g leche)',
    'mp_cuba': 'Materia proteica en cuba (g/100 g leche)',
    'lactosa_cuba': 'Contenido de lactosa en cuba (g/100 g leche)',
    'ph_final_moldeo_cuba': 'pH final antes del moldeo',
    'formato': 'Formato del queso una vez moldeado',
    'est_salida_salmuera': 'Extracto seco total tras salida de salado (g/100 g queso)',
    'mg_salida_salmuera': 'Materia grasa tras salida de salado (g/100 g queso)',
    'ph_salida_salmuera': 'pH tras salida de salado',
    'peso_queso_total': 'Peso total del queso producido (medido a la salida de la salmuera)',
    'sal_queso': 'Contenido de sal en el queso tras salida de salado (g/ 100 g queso)'
}

# Imprimir como tabla Markdown
# print("| Variable             | Descripción                                                            |")
# print("|----------------------|------------------------------------------------------------------------|")
# for variable, descripcion in descripcion_variables.items():
#     print(f"| {variable:<20} | {descripcion:<70} |")

# Construir tabla HTML
html = """
<table border="1">
    <thead>
        <tr>
           <th style="text-align:left;"><strong>Variable</strong></th>
           <th style="text-align:left;"><strong>Descripción</strong></th>
        </tr>
    </thead>
    <tbody>
"""

for variable, descripcion in descripcion_variables.items():
    html += f'        <tr><td style="text-align:left;">{variable}</td><td style="text-align:left;">{descripcion}</td></tr>\n'

html += "    </tbody>\n</table>"

# Mostrar en entorno compatible (como Jupyter Notebook)
from IPython.display import display, HTML
display(HTML(html))



Variable,Descripción
fecha,Fecha de producción
receta,Tipo de receta utilizada
litros_past,Litros de leche pasteurizada
est_past,Extracto seco total de la leche pasteurizada (g/100 ml leche)
mg_past,Materia grasa de la leche pasteurizada (g/100 ml leche)
mp_past,Materia proteica de la leche pasteurizada (g/100 ml leche)
kg_cuba,Kilos totales de leche en la cuba
est_cuba,Extracto seco total en cuba (g/100 g leche)
mg_cuba,Materia grasa en cuba (g/100 g leche)
mp_cuba,Materia proteica en cuba (g/100 g leche)


El laboratorio nos facillita la información analítica de la leche pasteurizada en porcentaje masa/volumen, mientas que la leche en cubas y el queso es masa/masa. Esto es así porque la cantidad de leche que pasa por el pasteurizador se mide con un contador volumétrico, mientras que en la cuba se mide con una célula de carga.

Si el volumen de leche en cuba se midiese también con contador volumétrico, necesitaríamos convertir los resultados analíticos a masa/volumen para obtener los kilos de cada materia, utilizando la densidad.

En la práctica, usar una densidad promedio fija o ajustada por estaciones funciona bastante bien, y evita los errores de medida de este parámetro que parecen sencillos pero siempre están sometidos al error de muestreo, además de la necesidad de corregir la temperatura.



Veamos los datos analíticos del queso

In [3]:
# Calcular media y mediana
media_est = df['est_salida_salmuera'].mean()
mediana_est = df['est_salida_salmuera'].median()

media_mg = df['mg_salida_salmuera'].mean()
mediana_mg = df['mg_salida_salmuera'].median()

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th><strong>Composición del queso</strong></th>
            <th><strong>Media (g/100 g)</strong></th>
            <th><strong>Mediana (g/100 g)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>EST a salida de salado</td>
            <td>{media_est:.2f}%</td>
            <td>{mediana_est:.2f}%</td>
        </tr>
        <tr>
            <td>MG a salida de salado</td>
            <td>{media_mg:.2f}%</td>
            <td>{mediana_mg:.2f}%</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar tabla en notebook
from IPython.display import display, HTML
display(HTML(html))

Composición del queso,Media (g/100 g),Mediana (g/100 g)
EST a salida de salado,55.27%,55.63%
MG a salida de salado,30.15%,30.05%


Para hacer este producto, estamos trabajando con una leche que hemos enriquecido en MG y MP, cuya composición es:

In [4]:
# Calcular medias y medianas desde df
media_est = df['est_cuba'].mean()*10
mediana_est = df['est_cuba'].median()*10

media_mg = df['mg_cuba'].mean()*10
mediana_mg = df['mg_cuba'].median()*10

media_mp = df['mp_cuba'].mean()*10
mediana_mp = df['mp_cuba'].median()*10

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th style="text-align:left;"><strong>Composición materia prima</strong></th>
            <th style="text-align:left;"><strong>Media (g/L)</strong></th>
            <th style="text-align:left;"><strong>Mediana (g/L)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="text-align:left;">Extracto seco total (ESM)</td>
            <td style="text-align:left;">{media_est:.2f}</td>
            <td style="text-align:left;">{mediana_est:.2f}</td>
        </tr>
        <tr>
            <td style="text-align:left;">Materia grasa (MG)</td>
            <td style="text-align:left;">{media_mg:.2f}</td>
            <td style="text-align:left;">{mediana_mg:.2f}</td>
        </tr>
        <tr>
            <td style="text-align:left;">Proteínas totales (MP)</td>
            <td style="text-align:left;">{media_mp:.2f}</td>
            <td style="text-align:left;">{mediana_mp:.2f}</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar en entorno compatible (como Jupyter Notebook)
from IPython.display import display, HTML
display(HTML(html))

Composición materia prima,Media (g/L),Mediana (g/L)
Extracto seco total (ESM),123.17,119.30
Materia grasa (MG),39.58,40.10
Proteínas totales (MP),33.84,33.50


## Cálculo de la cantidad de leche necesaria para fabricar 1 kg de queso

Sabemos que nuestro proceso de fabricación quesera consiste en la coagulación de la caseína y expulsión de suero, y que la materia grasa queda retenida en la red de caseína.

Este concepto es la base fundamental de la comprensión de la tecnología quesera: es el rendimiento en la recuperación de la proteína el que nos va a definir todo el proceso, la materia grasa “acompañará” de forma estática (aunque tendrá influencia en el desuerado)

Por esta razón, el principal elemento del rendimiento en la definición de la tecnología es el porcentaje de recuperación de proteína en el extracto seco magro (aislamos el efecto la grasa)

Hemos definido nuestro producto mediante los análisis de EST y MG; necesitamos saber el resto de parámetros, en concreto el ESM que estamos definiendo para nuestro producto:


In [5]:
# Calcular medias y medianas
media_est = df['est_salida_salmuera'].mean()
mediana_est = df['est_salida_salmuera'].median()

media_mg = df['mg_salida_salmuera'].mean()
mediana_mg = df['mg_salida_salmuera'].median()

# Calcular ESM (Extracto Seco Magro)
media_esm = media_est - media_mg
mediana_esm = mediana_est - mediana_mg

# Calcular índice MG/EST correctamente
media_mg_est = (media_mg / media_est) * 100
mediana_mg_est = (mediana_mg / mediana_est) * 100

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th style="text-align:left;"><strong>Composición del queso</strong></th>
            <th style="text-align:left;"><strong>Media (g/100 g)</strong></th>
            <th style="text-align:left;"><strong>Mediana (g/100 g)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="text-align:left;">EST a salida de salado</td>
            <td style="text-align:left;">{media_est:.2f}%</td>
            <td style="text-align:left;">{mediana_est:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">MG a salida de salado</td>
            <td style="text-align:left;">{media_mg:.2f}%</td>
            <td style="text-align:left;">{mediana_mg:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">ESM a salida de salado)</td>
            <td style="text-align:left;">{media_esm:.2f}%</td>
            <td style="text-align:left;">{mediana_esm:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">MG/EST a salida de salado</td>
            <td style="text-align:left;">{media_mg_est:.2f}%</td>
            <td style="text-align:left;">{mediana_mg_est:.2f}%</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar tabla en notebook
from IPython.display import display, HTML
display(HTML(html))

Composición del queso,Media (g/100 g),Mediana (g/100 g)
EST a salida de salado,55.27%,55.63%
MG a salida de salado,30.15%,30.05%
ESM a salida de salado),25.13%,25.58%
MG/EST a salida de salado,54.54%,54.02%


Ahora necesitamos conocer cuál es la tasa o ratio de conversión en ESM de queso del ESM, o bien de la MP, de la leche en cubas. Para ello calculamos estos ratios, que vamos a llamar **coeficiente de recuperación de ESM**, y el **coeficiente de recuperación de MP** junto con el **coeficiente de recuperación de MG**,que indicará la cantidad de MG que somos capaces de recuperar en el queso.

Si la recuperación de MG fuese perfecta, el coeficiente de recuperación sería 100%. El coeficiente refleja que hay pérdidas de MG, bien en el suero, bien en los finos de moldeo o en cuajada que se ha caído, etc, en resumen, que no ha ido a parar al queso moldeado. Podemos argumentar, correctamente, que la parte de los finos recoge también la eficiencia de la coagulación, pero al final lo que nos interesa es recoger la conversión real en queso de nuestra materia prima, excluyendo todo lo que constituyen las **pérdidas de proceso**.

El **coeficiente de recuperación de ESM** tiene un valor mucho más bajo, porque hay una parte significativa del ESM de la leche que vamos a perder en el suero y durante el salado, como las proteínas solubles, la lactosas y las sales minerales; por otra parte, vamos a incorporar a ese ESM la sal añadida durante el salado.

Lo importante es que el valor de este coeficiente, si nuestro proceso es estable, debe ser siempre constante: en un proceso estable, tanto las pérdidas como las ganancias deben ser muy semejantes de un día a otro, si las condiciones del proceso se han mantenido. Por lo tanto, lo que nos interesa es este valor *mediano* del coeficiente, y debemos asegurarnos de que es regular.

EL último coeficiente, el **coeficiente de recuperación de MP** se basa en esta misma regularidad del proceso para establecer la relación entre la proteína puesta en cuba y el ESM del queo. La ventaja de este coeficiente es que con los medios analíticos habituales, es más fácil analizar la leche mediante un NIR (*near-infrared*) como el MilkoScan, que hacer un extraco seco convencional en estufa.

In [6]:
### 1. Cálculos de cuba
df['est_kg_cuba'] = df['kg_cuba'] * df['est_cuba'] / 100
df['mg_kg_cuba'] = df['kg_cuba'] * df['mg_cuba'] / 100
df['mp_kg_cuba'] = df['kg_cuba'] * df['mp_cuba'] / 100
df['esm_kg_cuba'] = df['est_kg_cuba'] - df['mg_kg_cuba'] 

### 2. Cálculos de queso
df['est_kg_queso'] = df['peso_queso_total'] * df['est_salida_salmuera'] / 100
df['mg_kg_queso'] = df['peso_queso_total'] * df['mg_salida_salmuera'] / 100
df['esm_kg_queso'] = df['est_kg_queso'] - df['mg_kg_queso']
df['sal_kg_queso'] = df['peso_queso_total'] * df['sal_queso']/100

### 3. Coeficientes de recuperación (expresado en porcentaje)
df['coef_recup_mg']  = df['mg_kg_queso']/df['mg_kg_cuba'] * 100
df['coef_recup_mp']  = df['esm_kg_queso']/df['mp_kg_cuba'] * 100 # atencion a la fórmula de cálculo, deberia ser mp_kg_queso
df['coef_recup_esm'] = df['esm_kg_queso']/df['esm_kg_cuba'] * 100

print(f"Coeficiente de recuperación de materia grasa (MG queso/MG leche en cuba): {df['coef_recup_mg'].mean():.3f}")
print(f"Coeficiente de recuperación de proteína (ESM queso/MP leche en cuba): {df['coef_recup_mp'].mean():.3f}")
print(f"Coeficiente de recuperación de extracto seco magro (ESM queso/ESM leche en cuba): {df['coef_recup_esm'].mean():.3f}")




Coeficiente de recuperación de materia grasa (MG queso/MG leche en cuba): 94.178
Coeficiente de recuperación de proteína (ESM queso/MP leche en cuba): 92.498
Coeficiente de recuperación de extracto seco magro (ESM queso/ESM leche en cuba): 37.546


A partir de aquí, hacemos los cálculos teóricos del estándar técnico.

Utilizando el coeficiente de recuperación de la MP, podemos calcular la cantidad de ESM que vamos a recuperar. La mediana de la cantidad de proteína que tenemos en la leche es 33,50 g/L; sabemos que con 1 L de leche vamos a obtener en el queso

$$
\frac{33,50\ g\ proteína}{1\ L\ leche}*\ \frac{0.92498\ g\ ESM}{1\ g\ proteína} = 30,987\ g\ ESM\ en\ el\ queso\ por\ litro\ de\ leche
$$


Esta es la conversión clave de nuestra tecnología, y es conocida como ***coeficiente G*** en honor a su autor, Antoine M Guérault, que lo introdujo formalmente por primera vez por en su libro *La fromagerie devant les techniques nouvelles,* en 1966; desde entonces sigue utilizándose con plena validez. Originalmente, el coeficiente indica el porcentaje de ESM que hemos recuperado en el queso, dividiendo el ESM del queso entre el ESM de la leche puesta en fabricación. Como comentamos más atrás, para una tecnología concreta, normalmente la cantidad de sal incluida en el salado y las pérdidas de ESM en el proceso de fabricación suelen mantenerse en una horquilla regular, de acuerdo con la variabilidad de nuestro proceso y nuestra precisión analítica. Las diferencias en el valor del coeficiente nos indican si la transformación se ha realizado dentro de los valores normales de la tecnología o si hemos tenido valores anormalmente altos o bajos por lo que también usamos la proteína de la leche en vez del ESM. Esto lleva a una pequeña diferencia en el valor del coeficiente, que también se mantiene constante para la misma tecnología.

Como sabemos la cantidad de ESM que vamos a obtener de 1 L de leche, podemos calcular los litros de leche que necesitamos para obtener 1 kg de queso mediante una regla de tres simple: si para obtener 100 g de queso con 30,987 g de ESM necesitamos 1 L de leche, para obtener 1 kg de queso con el mismo ESM necesitaremos X litros de leche. A este valor lo llamaremos *litraje unitario*.

$$
\text{Litraje unitario} = \frac{25,58\ g\ ESM}{100\ g\ queso\ }*\ \frac{1000\ g\ queso}{1\ kg\ queso}*\ \frac{1\ L\ leche}{30,987\ g\ ESM} = 8,255\ L\ leche\ para\ hacer\ 1\ kg\ de\ queso
$$

y el consumo de MP por kilo de queso es el que resulta de calcular

$$
8,255\ L\ leche*\ \frac{33,50\ g\ MP}{1\ L\ leche} = 276,543\ g\ MP\ por\ kilo\ de\ queso
$$


Es necesario insistir en que estos cálculos sólo son válidos en nuestras condiciones de trabajo precisas:

-   con la composición en ESM de nuestro queso

-   con la composición de proteína de nuestra leche

-   con la tasa de recuperación de la proteína de la leche en el ESM del queso de nuestro específico proceso de fabricación.

Son valores teóricos que utilizamos como valor objetivo, y que pueden variar a medida que adaptemos o variemos nuestra tecnología. Para cualquier otro producto, otra composición de leche y/u otro proceso de fabricación, deberemos recalcular nuestro litraje por kilo y nuestro consumo,

Sabemos que, en nuestras condiciones, 8,255 L de leche deben darnos 1 kg de queso. Si obtenemos más o menos queso querrá decir que nuestras hipótesis de partida no se han cumplido, y analizando los tres elementos básicos, sabremos la razón de la desviación:

-   Nuestra leche tiene una composición diferente

-   Nuestro queso tiene una composición diferente

-   Nuestra recuperación de proteína ha sido diferente.

Si disponemos de la analítica correspondiente, podremos tomar las decisiones correctas; no es lo mismo actuar ante un problema de composición de leche que ante un problema de composición de queso o un problema de tecnología quesera (el extracto seco magro del queso es consecuencia de la tecnología quesera). Otros factores pueden influir directamente en el coeficiente de recuperación de proteína, por ejemplo, la cantidad de proteína coagulable (caseína) frente a la proteína total analizada puede variar a lo largo del año, y el análisis de proteína total no nos da una indicación de esta ratio.

## La materia grasa

Hasta aquí hemos hablado de proteína y ESM, pero ¿y la materia grasa?

En nuestro análisis de rendimiento hemos calculado también un % de recuperación de MG. Sin embargo, no utilizaremos este % para calcular el litraje, lo haremos tal como hemos visto, con la transformación de la proteína en ESM. La razón es que *la tecnología quesera nos permite actuar sobre el ESM del queso, pero la MG depende de la estandarización*, como veremos a continuación.

Como el cálculo que hemos realizado con el coeficiente de recuperación de la MP nos da los litros de leche que necesitamos para hacer 1 kg de queso, y tenemos una composición de nuestra leche, podemos saber la cantidad de MG que vamos a tener en esos litros. Por otra parte, sabemos la MG que hemos definido en nuestro queso de forma empírica, es decir, en la práctica: mediante un promedio de los diferentes análisis de nuestro producto que hemos ido obteniendo.

Hemos calculado nuestro litraje unitario: necesitamos 8,255 L leche para hacer 1 kg de queso.

Según nuestra composición de leche, 1 L de leche tiene 40,10 g de MG, luego los 8,255 L de leche tendrán

$$
8,255\ L\ leche*\ \frac{40,10\ g\ MG}{1\ L\ leche} = 331,026\ g\ MG
$$

Es decir, para fabricar 1 kg de queso necesitamos 8.255 L leche, que según su composición tienen 331,026 g MG

Sin embargo, la analítica de nuestro queso nos dice que en 1 kg de queso tenemos 30,05% de MG, es decir 300,5 g de MG por kilo de queso.

¿A qué se debe esta diferencia? La respuesta es: *a las pérdidas de MG en el proceso*, es decir, *a nuestro porcentaje de recuperación de MG.*

Podemos calcular este porcentaje mediante la fracción

$$
\frac{MG\ en\ el\ queso}{MG\ en\ la\ leche\ de\ fabricación}\ =\ \frac{300,5\ g\ MG/kg\ en\ el\ queso}{331,026\ g\ MG/kg\ en\ la\ leche\ de\ fabricación} = 0,9078
$$

Es decir, nuestra recuperación teórica de MG es del 90.78%

¿Por qué *recuperación teórica*? Porque es la que resulta de hacer los cálculos según nuestros datos de composición de leche, composición del queso y recuperación de proteína.

De la misma forma que en el caso de la recuperación de proteína, si nuestra recuperación de MG es diferente de la calculada teóricamente, debemos encontrar la explicación en los mismos factores:

-   Composición real de la leche

-   Composición real del queso

-   Variaciones en la tecnología

Estas variaciones implicarán una mayor pérdida de MG, que se irá en el suero de quesería, y que en parte puede recuperarse mediante un desnatado del suero.

## El estándar de fabricación

El razonamiento y cálculos que hemos ido realizando, se suelen recoger en lo que se llama un *estándar técnico de fabricación*. Este documento recoge los valores estándar de nuestra tecnología dada una composición media de nuestro queso obtenido (o un objetivo a obtener) y una composición de leche de partida. Al final del capítulo se puede descargar una hoja Excel con el estándar de fabricacion que hemos ido construyendo para nuestro queso de vaca.

Llamamos **MG teórica en la cuba** a la cantidad de MG que calculamos que vamos a tener en la cuba, es decir, en nuestros 8,255 L de leche a fabricar. Esta MG puede provenir de la leche de partida o del resultado de añadir nata a la leche para conseguir la cantidad necesaria.

De la misma forma, llamamos **MP en cuba** a la cantidad de MP que hemos puesto en cuba, ya sea la de la leche de partida o del resultado de haber añadido proteínas a esta leche.

## El estándar técnico final

Como hemos visto, la construcción del estándar técnico y el seguimiento de la tecnología quesera no se hacen siguiendo el consumo de *litros de leche*, sino de la cantidad de proteína y grasa utilizadas. Por esta razón, la construcción del estándar debe proporcionar las cantidades de MG y MP necesarias para obtener 1 kg de queso, y no sólo el litraje en cuba

En este caso, vemos un proceso de fabricación que *no* está recuperando la materia grasa del suero; las diferencias entre la MG en cuba y la MG en el queso se consideran pérdidas de proceso.

## La importancia de la estandarización de la leche

La tecnología quesera se diseña para optimizar el coeficiente de recuperación de proteína. Aunque la retención de materia grasa (MG) también depende de factores tecnológicos como el corte, el calentamiento y el pH, no está directamente controlada por la tecnología, aunque sí influenciada por ella.

En una tecnología concreta, con parámetros de trabajo fijados, cuando la cantidad de MG en la leche es superior a la definida en el proceso, se producen dos efectos:

-   Una parte del exceso de MG queda retenida en la matriz proteica, lo que incrementa la cantidad de grasa en el queso y, por tanto, modifica su composición.

-   Otra parte de la MG se pierde en el suero, ya que las condiciones del proceso determinan una relación fija con la cantidad total de MG de partida. Es decir, al aumentar la MG inicial, también aumentan las pérdidas relativas en el suero.

Desgraciadamente, no es posible calcular teóricamente cuánta MG se retendrá en el queso y cuánta se perderá en el suero, ya que estas cantidades no responden a una función conocida. Lo que sí sabemos es que:

-   Aumentarán las pérdidas.

-   Se modificará la composición del queso, lo que puede alterar su textura y el valor nutricional declarado.

-   Se alterará la expulsión de suero de la matriz proteica, lo que afectará la cantidad de lactosa retenida en la cuajada y, en consecuencia, las curvas de acidificación, el extracto seco, etc.

Cuando la MG es *inferior* a la definida, estos efectos se producen en sentido inverso, afectando tanto la composición del queso como su comportamiento tecnológico.

Para minimizar estas variaciones y mantener una composición constante del producto, es necesario:

-   Trabajar con leche que tenga un contenido estable de materia grasa (MG) y materia proteica (MP). Dado que la composición de la leche varía estacionalmente, esto implica estandarizarla en MG y MP, *manteniendo constante la relación MG/MP*.

-   Adaptar la tecnología de forma estacional, ya que la variación estacional no solo afecta la relación MG/MP, sino también la proporción de caseína coagulable dentro del total de proteínas.

Es fundamental recordar lo señalado al definir el estándar técnico: *la tecnología permite modificar el extracto seco magro (ESM) del queso, pero la MG depende principalmente de la cantidad presente en la leche de fabricación*. **No es posible modificar la cantidad de grasa en el queso mediante la tecnología quesera en la cuba**; es necesario preparar la leche con la cantidad de grasa adecuada para cubrir tanto la retención esperada como las pérdidas previstas.

En la medida en que no se puedan realizar estas adaptaciones, se deberá aceptar una mayor variabilidad en el producto, con dos consecuencias:

-   Irregularidad en el producto final, que afecta la aceptación por parte del consumidor.

-   Variación en el coste del producto, que puede generar pérdidas no previstas.

Por estas razones, es imprescindible realizar un seguimiento detallado de las desviaciones tecnológicas, para comprender mejor el comportamiento del proceso y actuar eficazmente sobre sus causas, estableciendo los planes correctivos oportunos.

En una pestaña adicional a la hoja Excel del estándar técnico, se incluye una tabla con un estándar  teórico y un resultado real, para comparar los diferentes valores. El supuesto real parte de una fabricación hipotética de 105 L de leche, con los que hemos obtenido 11,2 kg de queso; el teórico se construye a partir del estándar técnico, aplicando los coeficientes correspondientes a esa cantidad, lo que permite calcular los valores teóricos esperados. Los cálculos se realizan de diferente manera, para llegar a los resultados comparables: el estándar teórico se calcula como hemos ido viendo aquí, el real se calcula con los datos reales del queso fabricado y la materia prima utilizada. El análisis de desviaciones incluye una estimación del coste de las desviaciones a partir de un precio de leche supuesto.

Las desviaciones observadas deben analizarse con base en el conocimiento tecnológico, para establecer planes correctivos adecuados.

Conociendo el coste de la MG y de la MP pagada, es posible calcular el valor económico de cada desviación, lo que permite priorizar las acciones correctivas según su impacto financiero.

Hay una hoja Excel con todos los cálculos, que incluye también los estándares de fabricacion que veremos en el capítulo siguiente para el resto de recetas.


[Abrir hoja Excel con modelo](https://raw.githubusercontent.com/juanriera/master-queseria/master/datos/2025-11-06-ejemplo-std-tecnico.xlsx){target="_blank" rel="noopener noreferrer"}